In [10]:
import numpy as np
import matplotlib.pyplot as plt

from sko.GA import GA

from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.indicators.hv import HV

In [ ]:
def plot_compromisso_8_funcoes(
    solucoes,
    nomes_funcoes=None,
    titulo="Compromisso entre 8 funções objetivo",
    escalas=None
):
    """
    Radar chart de 8 resíduos.
    Destaca a solução minimax e exibe apenas sua legenda.

    solucoes: array (n_solucoes, 8) ou dict {nome: residuos}
    escalas: escalas de normalização, opcional.
    """

    if isinstance(solucoes, dict):
        nomes = list(solucoes.keys())
        valores = np.array(list(solucoes.values()), dtype=float)
    else:
        valores = np.asarray(solucoes, dtype=float)
        if valores.ndim == 1:
            valores = valores.reshape(1, -1)
        nomes = [f"Solução {i+1}" for i in range(len(valores))]

    if valores.shape[1] != 8:
        raise ValueError("Cada solução deve ter exatamente 8 resíduos.")

    valores = np.abs(valores)

    if escalas is not None:
        escalas = np.asarray(escalas, dtype=float)
        if len(escalas) != 8 or np.any(escalas <= 0):
            raise ValueError("Informe 8 escalas positivas.")
        valores_norm = valores / escalas
    else:
        valores_norm = valores.copy()

    # Seleciona a solução minimax: menor pior resíduo normalizado.
    escores = np.max(valores_norm, axis=1)
    idx_best = np.argmin(escores)

    if nomes_funcoes is None:
        nomes_funcoes = [f"f{i+1}" for i in range(8)]

    angulos = np.linspace(0, 2*np.pi, 8, endpoint=False)
    angulos_fechados = np.append(angulos, angulos[0])

    fig, ax = plt.subplots(
        figsize=(8, 8),
        subplot_kw={"polar": True}
    )

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angulos)
    ax.set_xticklabels(nomes_funcoes)

    # Outras soluções em cinza, sem legenda.
    for i, residuos in enumerate(valores_norm):
        if i == idx_best:
            continue

        dados = np.append(residuos, residuos[0])
        ax.plot(
            angulos_fechados, dados,
            color="gray", alpha=0.35, linewidth=1
        )

    # Solução minimax destacada, com a única legenda.
    melhor = valores_norm[idx_best]
    dados_best = np.append(melhor, melhor[0])

    ax.plot(
        angulos_fechados, dados_best,
        color="red", linewidth=2.5,
        label="Melhor solução (minimax)"
    )
    ax.fill(
        angulos_fechados, dados_best,
        color="red", alpha=0.12
    )

    maior = np.max(valores_norm)
    ax.set_ylim(0, maior * 1.15 if maior > 0 else 1)

    ax.set_title(titulo, pad=25)
    ax.legend(loc="upper left", bbox_to_anchor=(1.05, 1.1))

    plt.tight_layout()
    plt.show()


In [11]:
fun_1 = [
    lambda x: 0.8*(x[0]**2 + x[0] -  1)*x[2] + 0.12*x[0] + 2.16*x[0] - 0.12,
    lambda x: (1 + x[0]**2)*x[3] + 0.4*x[0]**2 - 1.6*x[0] - 0.4,
    lambda x: (1 + x[0]**2)*x[4] + x[0]**2 - 1,
    lambda x: (1 + x[0]**2)*x[5] + 0.8*(x[0]**2 + x[0] - 1),
    lambda x: x[2]*x[6] - 0.02*x[5] - x[4] - x[2]*x[3] - 0.16*x[3],
    lambda x: x[6]**2 - 2*x[3]*x[6] + x[5]**2 + x[3]**2 - x[1]**2,
    lambda x: x[7] - x[1]*x[2],
    lambda x: 0.0476*x[2]*x[7]**12 + x[2] - 2.104
]

fun_2 = [
    lambda x: x[4] + x[3] - 1.803,
    lambda x: (x[1] + x[2])*x[4] + 6.19116*x[3] - 1.803*(1.497 + 0.035),
    lambda x: x[5] + x[3] - 0.328,
    lambda x: 0.28801*x[5] - x[1]*x[2]*x[4],
    lambda x: (-6.19116*x[0] + x[0]*x[2] + x[1]*x[4] - x[2]*x[4])*x[5] + x[0]*x[2]*x[4],
    lambda x: 1.571*x[6] + x[3] - 1.803,
    lambda x: x[7] - 0.000856*x[6]**2,
    lambda x: (x[4] - x[0])*x[8] - x[0]*x[4],
    lambda x: x[8] - 377*x[1]*x[7]
]


In [12]:
def eval_fun(fun_list, x):
    return [f(x) for f in fun_list]

In [13]:
lb_1 = [-3, -1, -2, -1, -1, -0.5, -1.5, -1.5]
ub_1 = [1, 1, 2, 1, 1, 0.5, 1.5, 1.5]

lb_2 = [-0.5, -1, -1, -1, 1, -1, -1, 0, -1]
ub_2 = [0.5, 1, 1, 1, 2, 1, 1, 1, 1]

In [ ]:
def get_escala(lista_funcoes, lb, ub, num_amostras_escala):
    parametros_aleatorios = np.random.uniform(lb, ub, size=(num_amostras_escala, len(lb)))
    escalas = [np.std([f(param) for param in parametros_aleatorios]) for f in lista_funcoes]

    return escalas

def get_solucoes_normalizadas(historico_x, lista_funcoes, lb, ub, num_amostras_escala=100000):
    escalas = get_escala(lista_funcoes, lb, ub, num_amostras_escala)

    resultados_best_params = np.array([[f(param) for f in lista_funcoes] for param in historico_x])

    resultados_normalizados = resultados_best_params/escalas

    return resultados_normalizados

def melhor_solucao_minimax(historico_x, lista_funcoes, lb, ub, num_amostras_escala=100000):
    resultados_normalizados = get_solucoes_normalizadas(historico_x, lista_funcoes, lb, ub, num_amostras_escala)

    piores_resultados = [np.max(np.abs(r)) for r in resultados_normalizados]

    return historico_x[np.argmin(piores_resultados)]

In [16]:
historico = []
for n in range(30):
    ga = GA(
    func=lambda x: 0.8*(x[0]**2 + x[0] -  1)*x[2] + 0.12*x[0] + 2.16*x[0] - 0.12,
    n_dim=8,
    size_pop=100,
    max_iter=1000,
    prob_mut=0.1,
    lb=lb_1,
    ub=ub_1,
    constraint_eq=fun_1[1:]
)
    
    best_x, best_y = ga.run()
    historico.append(best_x)

In [55]:
# plot_compromisso_8_funcoes(solucoes=[eval_fun(fun_1, sol) for sol in historico], escalas=get_escala(fun_1, lb_1, ub_1, 100000))

In [ ]:
class Problema1(ElementwiseProblem):

    def __init__(self):
        super().__init__(
            n_var=8,      
            n_obj=8,      
            xl=lb_1,
            xu=ub_1
        )

    def _evaluate(self, x, out, *args, **kwargs):

        out["F"] = [abs(f(x)) for f in fun_1]


class Problema2(ElementwiseProblem):

    def __init__(self):
        super().__init__(
            n_var=9,      
            n_obj=9,      
            xl=lb_2,
            xu=ub_2
        )

    def _evaluate(self, x, out, *args, **kwargs):

        out["F"] = [abs(f(x)) for f in fun_2]

In [9]:
problem1 = Problema1()
problem2 = Problema2()

algorithm = NSGA2(
    pop_size=100,

    sampling=FloatRandomSampling(),

    crossover=SBX(
        prob=0.6,
        eta=15
    ),

    mutation=PM(
        prob=1.0,
        prob_var = 1 / problem1.n_var,
        eta=20
    ),

    eliminate_duplicates=True
)

result1 = minimize(
    problem1,
    algorithm,
    termination=("n_gen", 1500),
    verbose=False,
    save_history=True
)

# result2 = minimize(
#     problem2,
#     algorithm,
#     termination=("n_gen", 1500),
#     seed=42,
#     verbose=False
# )

In [ ]:
# def pond_obj_1(x):
#     objs = obj_fun_1(x[:8])

#     zip_ob = zip(objs, x[8:])

#     return abs(sum([f*w for f,w  in list(zip_ob)]))

# def pond_obj_2(x):
#     objs = obj_fun_2(x[:8])

#     zip_ob = zip(objs, x[8:])

#     return abs(sum([f*w for f,w  in list(zip_ob)]))

# ga_test = GA(
#     func=pond_obj_1,
#     n_dim=16,
#     size_pop=100,
#     max_iter=1000,
#     lb=lb_1 + [-1]*8,
#     ub=ub_1 + [1]*8
# )